# RAG Evaluation with RAGAS + Gemini (generation) + Groq (judge)

This notebook builds a small RAG pipeline (Qdrant + Gemini) and evaluates it with
[RAGAS](https://docs.ragas.io). **Generation** (the RAG pipeline that answers
questions) still uses Google Gemini. The **RAGAS judge LLM** (the model that
scores Faithfulness / Response Relevancy) now uses **Groq** instead of Gemini,
because Groq's free tier has a much higher requests-per-minute allowance than
Gemini's free tier, which is what was actually causing the 429 errors even
after adding `asyncio.sleep()` calls (see the markdown cell right before
section 10 for the full explanation).

**Fixed in this version:**
- Removed duplicate / conflicting `pip install` cells and pinned versions that
  are known to install together without dependency conflicts.
- Fixed the incomplete `build_prompt` function (was missing `:` and a body,
  which caused a `SyntaxError` when the cell ran).
- Removed the unused OpenAI imports/functions (`langchain_openai`, `openai`)
  since the RAG pipeline runs on Gemini only.
- Consolidated the many repeated "RAGAS LLM/embeddings" cells into a single
  definition.
- Fixed `list(client.list_examples(...))[0]` being called three separate times
  (three separate API calls) → now called once and reused.
- Added `nest_asyncio` so the `await` calls work reliably inside Jupyter no
  matter which kernel/event loop is running.
- **Swapped the RAGAS judge LLM from Gemini to Groq** and removed the
  trial-and-error `asyncio.sleep(5/40/60)` cells that were papering over a
  problem sleeping can't fix (explained in section 10).


## 1. Install dependencies

These versions were verified together in a clean virtual environment
(`pip check` reports **no broken requirements**):

| package | version |
|---|---|
| ragas | 0.3.7 |
| langchain | 0.3.30 |
| langchain-core | 0.3.86 |
| langchain-community | 0.3.31 |
| langchain-google-genai | 2.1.12 |
| langchain-groq | 0.3.8 |
| google-genai | latest 2.x |
| qdrant-client | latest 1.x |
| langsmith | latest |
| python-dotenv | latest |

> ⚠️ Do **not** `pip install -U langchain langchain-google-genai` on top of this.
> The newest `langchain-google-genai` (4.x) requires `langchain-core>=1.4`,
> which is incompatible with `ragas==0.3.7` (which still depends on the
> `langchain-community` 0.3.x line). Mixing the two is what caused the
> `ModuleNotFoundError: No module named 'langchain_community.chat_models.vertexai'`
> style errors in the original notebook.
>
> `langchain-groq==0.3.8` needs `langchain-core>=0.3.75,<1.0.0`, which fits the
> pinned `langchain-core==0.3.86` above — no conflict with the rest of the stack.


In [1]:
! uv pip install -q "ragas==0.3.7" \
    "langchain==0.3.30" \
    "langchain-core==0.3.86" \
    "langchain-community==0.3.31" \
    "langchain-google-genai==2.1.12" \
    "langchain-groq==0.3.8" \
    "google-genai" \
    "qdrant-client" \
    "langsmith" \
    "python-dotenv" \
    "nest_asyncio"


## 2. Imports

In [2]:
import os
import asyncio

import nest_asyncio
from dotenv import load_dotenv

# Google & Qdrant
from google import genai
from google.genai import types
from qdrant_client import QdrantClient

# LangSmith
from langsmith import Client

# LangChain (Gemini) wrapper used by the RAG pipeline's generation step
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# LangChain (Groq) wrapper used ONLY as the RAGAS judge LLM
from langchain_groq import ChatGroq

# RAGAS
from ragas import SingleTurnSample
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import (
    Faithfulness,
    ResponseRelevancy,
    IDBasedContextPrecision,
    IDBasedContextRecall,
)

# Lets `await` work smoothly inside Jupyter regardless of the kernel's event loop
nest_asyncio.apply()


2026-07-08 23:21:47.268047009 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


## 3. Environment & clients

Put a `.env` file next to this notebook containing:

```
GOOGLE_API_KEY=your-gemini-api-key
GROQ_API_KEY=your-groq-api-key
LANGSMITH_API_KEY=your-langsmith-api-key
```

`GOOGLE_API_KEY` is used for the RAG pipeline (generation + embeddings).
`GROQ_API_KEY` is used only for the RAGAS judge LLM (section 5 below).


In [3]:
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
    raise ValueError("GOOGLE_API_KEY not found in .env file!")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in .env file!")

# Native google-genai client -> used for the RAG pipeline itself
# (embeddings + answer generation)
gemini_client = genai.Client(api_key=GOOGLE_API_KEY)

# LangSmith client -> used to pull the evaluation dataset
ls_client = Client()


## 4. Shared model configuration

**Generation stays on Gemini. The RAGAS judge moves to Groq.** Two things worth knowing:

1. **Why the judge moved providers, not just models.** Sharing one model for
   generation *and* evaluation only has a mild "self-preference bias" (a
   model rates its own answers slightly more favorably). That's a quality
   concern, not the bug here — see the markdown cell in section 10 for what
   actually caused the persistent 429 errors regardless of `asyncio.sleep()`.
   Using Groq as the judge sidesteps that root cause because Groq's free-tier
   requests-per-minute allowance is well above what four sequential RAGAS
   calls need, whereas Gemini's free tier is not.
2. **Judge model note:** `llama-3.3-70b-versatile` is being deprecated by Groq
   (announced June 17, 2026). This notebook uses `openai/gpt-oss-120b`, Groq's
   current recommended general-purpose replacement. Check
   https://console.groq.com/docs/models for the latest lineup before relying
   on this in a long-lived project — Groq's free-tier catalog and limits
   change fairly often.
3. **Embeddings stay on Gemini.** Groq doesn't serve an embeddings endpoint,
   so `ResponseRelevancy` (which needs embeddings, not just an LLM) keeps
   using the same `EMBEDDING_MODEL` as Qdrant retrieval — no change there.


In [4]:
# ---------------------------------------------------------------
# Single source of truth for every model name used in this notebook.
# ---------------------------------------------------------------
GENERATION_MODEL = "gemini-2.5-flash"     # used by the RAG pipeline to answer questions (Gemini)
EMBEDDING_MODEL = "gemini-embedding-001"  # used for BOTH Qdrant retrieval and RAGAS ResponseRelevancy
                                           # (models/embedding-001 and models/text-embedding-004
                                           # were retired by Google on Jan 14, 2026 - do not use them)

JUDGE_MODEL = "openai/gpt-oss-120b"       # used by RAGAS to score faithfulness/relevancy (Groq)
                                           # llama-3.3-70b-versatile is being deprecated by Groq -
                                           # check https://console.groq.com/docs/models if this stops working


## 5. RAGAS evaluator LLM & embeddings


In [5]:
# RAGAS needs its own LLM (to judge faithfulness / relevancy) and its own
# embedding model (for ResponseRelevancy).
#
# - Judge LLM -> Groq (ChatGroq), NOT Gemini. This is the actual fix for the
#   429 errors: Groq's free tier gives far more requests/minute than Gemini's,
#   so four sequential judge calls no longer collide with the rate limit.
# - Embeddings -> stay on Gemini, since Groq has no embeddings endpoint.
ragas_llm = LangchainLLMWrapper(
    ChatGroq(model=JUDGE_MODEL, groq_api_key=GROQ_API_KEY)
)

ragas_embeddings = LangchainEmbeddingsWrapper(
    GoogleGenerativeAIEmbeddings(model=f"models/{EMBEDDING_MODEL}", google_api_key=GOOGLE_API_KEY)
)


/tmp/ipykernel_3822553/4076128048.py:8: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use the modern LLM providers instead: from ragas.llms.base import llm_factory; llm = llm_factory('gpt-4o-mini') or from ragas.llms.base import instructor_llm_factory; llm = instructor_llm_factory('openai', client=openai_client)
  ragas_llm = LangchainLLMWrapper(
/tmp/ipykernel_3822553/4076128048.py:12: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(


## 6. Download a reference example from LangSmith


In [6]:
dataset = ls_client.read_dataset(dataset_name="rag-evaluation-dataset")

# Pull the examples ONCE and reuse them (the original notebook called
# list_examples three separate times for no reason)
examples = list(ls_client.list_examples(dataset_id=dataset.id, limit=10))

reference_input = examples[0].inputs
reference_output = examples[0].outputs

reference_input, reference_output


({'question': 'What is the warranty period for the Garmin 890 RV GPS Navigator?'},
 {'ground_truth': "I'm sorry, but the specific duration of the warranty for the Garmin 890 RV GPS Navigator is not mentioned in the product description, only that it comes with a 'Limited Warranty'.",
  'reference_context_ids': [],
  'reference_descriptions': []})

## 7. RAG pipeline

* `get_embedding` – embeds a query with the shared `EMBEDDING_MODEL`
* `retrieve_data` – queries Qdrant and returns ids/context/ratings/scores
* `process_context` – turns retrieved chunks into a formatted context string
* `build_prompt` – builds the final prompt for the answer-generation model
* `generate_answer` – calls Gemini (`GENERATION_MODEL`) to answer the question
* `rag_pipeline` – glues everything together


In [7]:
def get_embedding(text, model=EMBEDDING_MODEL):
    """Generate an embedding vector for `text` using a Gemini embedding model."""
    response = gemini_client.models.embed_content(
        model=model,
        contents=text,
        config=types.EmbedContentConfig(),
    )
    return response.embeddings[0].values


def retrieve_data(query, qdrant_client, k=5):
    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-03",
        query=query_embedding,
        limit=k,
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["description"])
        retrieved_context_ratings.append(result.payload["average_rating"])
        similarity_scores.append(result.score)

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "retrieved_context_ratings": retrieved_context_ratings,
        "similarity_scores": similarity_scores,
    }


def process_context(context):
    formatted_context = ""
    for id_, chunk, rating in zip(
        context["retrieved_context_ids"],
        context["retrieved_context"],
        context["retrieved_context_ratings"],
    ):
        formatted_context += f"- ID: {id_}, rating: {rating}, description: {chunk}\n"
    return formatted_context


def build_prompt(preprocessed_context, question):
    """NOTE: this function was left incomplete (missing `:` and body) in the
    original notebook, which raised a SyntaxError. Fixed here."""
    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- You need to answer the question based on the provided context only.
- Never use the word "context" and refer to it as the available products.

Context:
{preprocessed_context}

Question:
{question}
"""
    return prompt


def generate_answer(prompt, model_name=GENERATION_MODEL):
    """Generates a response using the specified Gemini model."""
    response = gemini_client.models.generate_content(
        model=model_name,
        contents=prompt,
    )
    return response.text


def rag_pipeline(question, top_k=5):
    qdrant_client = QdrantClient(url="http://localhost:6333")

    retrieved_context = retrieve_data(question, qdrant_client, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    return {
        "answer": answer,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"],
        "similarity_scores": retrieved_context["similarity_scores"],
    }


### Quick smoke test

In [8]:
rag_pipeline("Can I get some charger?", top_k=5)


{'answer': "Yes, there are several charging options available:\n\n*   **For Fitbit Inspire 3:** There is a Mixblu Charger Cable Replacement (2 pack/3.3ft) specifically designed for this model, offering fast and stable charging.\n*   **For multiple devices (iPhone, Android, Kindle, etc.):** A 5-in-1 USB C to Multi Charging Cable is available. It includes USB A/USB C to Lightning, Type C, and Micro USB connectors, allowing you to charge three devices simultaneously. It's 10ft long and Apple MFi Certified, but note that the USB C port does not work with iPads, tablets, or larger power-requirement devices.\n*   **For Apple Lightning devices (iPhone, iPad, iPod, AirPods):**\n    *   **USB A to Lightning cables:**\n        *   An original Apple MFi Certified iPhone Charger Cord (3-pack, 3ft) for fast charging and data transfer.\n        *   A MUXA 6-pack of Apple MFi Certified colorful nylon lightning cables in various lengths (3/3/6/6/10/10 FT) for iPhone, iPad, and iPod.\n    *   **USB C t

## 8. Run the pipeline on the LangSmith reference question


In [9]:
result = rag_pipeline(reference_input["question"])
result


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 18.835750394s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-2.5-flash', 'location': 'global'},

In [9]:
result = rag_pipeline(reference_input["question"])
result


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 46.212251997s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'},

In [10]:
result = rag_pipeline(reference_input["question"])
result


{'answer': 'The Garmin 890 RV GPS Navigator comes with a limited warranty. The available products do not specify the exact duration of the warranty.',
 'question': 'What is the warranty period for the Garmin 890 RV GPS Navigator?',
 'retrieved_context_ids': ['B08BX2L8F2',
  'B0B3MMP22L',
  'B0BLV7YH2W',
  'B0BB1YKXL1',
  'B09QGNB537'],
 'retrieved_context': ['Garmin 890 8-inch RV GPS Navigator Bundle with Car Charger Expander and Hard Shell EVA Case for Tablets/GPS (010-02425-00) Built in Wi-Fi connectivity makes updating your maps of North America a breeze. This large 8" GPS navigator features a bright, high-resolution edge-to-edge touchscreen display so you can easily see important information Built-in Wi-Fi connectivity makes it easy to keep your maps and software up to date without using a computer IN THE BOX: RV 890 | Vehicle suction cup with powered magnetic mount | Screw down mount | 1" ball adapter with AMPS plate | Vehicle power cable | USB cable | Documentation | Limited Warr

In [8]:
result = rag_pipeline(reference_input["question"])
result


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 7.513570017s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-2.5-flash', 'location': 'global'}, 

In [12]:
result = rag_pipeline(reference_input["question"])
result


{'answer': 'The available products mention that the Garmin 890 8-inch RV GPS Navigator comes with a "Limited Warranty," but they do not specify the duration of this warranty.',
 'question': 'What is the warranty period for the Garmin 890 RV GPS Navigator?',
 'retrieved_context_ids': ['B08BX2L8F2',
  'B0B3MMP22L',
  'B0BLV7YH2W',
  'B0BB1YKXL1',
  'B09QGNB537'],
 'retrieved_context': ['Garmin 890 8-inch RV GPS Navigator Bundle with Car Charger Expander and Hard Shell EVA Case for Tablets/GPS (010-02425-00) Built in Wi-Fi connectivity makes updating your maps of North America a breeze. This large 8" GPS navigator features a bright, high-resolution edge-to-edge touchscreen display so you can easily see important information Built-in Wi-Fi connectivity makes it easy to keep your maps and software up to date without using a computer IN THE BOX: RV 890 | Vehicle suction cup with powered magnetic mount | Screw down mount | 1" ball adapter with AMPS plate | Vehicle power cable | USB cable | Do

## 9. RAGAS metrics

Four metrics are used:

- **Faithfulness** – is the answer grounded in the retrieved context? (needs an LLM)
- **ResponseRelevancy** – is the answer relevant to the question? (needs an LLM + embeddings)
- **IDBasedContextPrecision** – of the retrieved ids, how many are in the reference ids? (no LLM needed, pure ID comparison)
- **IDBasedContextRecall** – of the reference ids, how many were retrieved? (no LLM needed, pure ID comparison)


In [ ]:
async def ragas_faithfulness(run, example):
    sample = SingleTurnSample(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_context"],
    )
    scorer = Faithfulness(llm=ragas_llm)
    return await scorer.single_turn_ascore(sample)


# async def ragas_response_relevancy(run, example):
#     sample = SingleTurnSample(
#         user_input=run["question"],
#         response=run["answer"],
#         retrieved_contexts=run["retrieved_context"],
#     )
#     scorer = ResponseRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)
#     return await scorer.single_turn_ascore(sample)



async def ragas_response_relevancy(run, example):
    sample = SingleTurnSample(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_context"]
    )
    
    # THE FIX: Add strictness=1 so Ragas passes n=1 to Groq
    scorer = ResponseRelevancy(
        llm=ragas_llm, 
        embeddings=ragas_embeddings,
        strictness=1 
    )
    
    return await scorer.single_turn_ascore(sample, run_config=safe_config)


async def ragas_context_precision_id_based(run, example):
    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"],
    )
    scorer = IDBasedContextPrecision()
    return await scorer.single_turn_ascore(sample)


async def ragas_context_recall_id_based(run, example):
    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"],
    )
    scorer = IDBasedContextRecall()
    return await scorer.single_turn_ascore(sample)


## 10. Run all evaluations

### Why `asyncio.sleep(5)` / `sleep(40)` / `sleep(60)` never fixed the 429s

Three earlier attempts in this notebook tried fixed delays *between* the four
metric calls (5s, then 40s, then 60s). None of them could have worked
reliably, for reasons that have nothing to do with how long the sleep was:

1. **The sleep was in the wrong place.** `Faithfulness.single_turn_ascore()`
   doesn't make one LLM call — internally RAGAS first calls the judge LLM to
   *decompose* the answer into individual statements, then calls it again to
   *verify* each statement against the retrieved context. That's 2+ calls
   fired back-to-back **inside a single `await ragas_faithfulness(...)` line**,
   before your `asyncio.sleep()` between metrics ever runs. A 60-second gap
   between metrics does nothing to space out the calls that happen inside one
   metric.
2. **Free-tier quotas aren't only per-minute.** Gemini's free tier enforces
   RPM (requests/minute), TPM (tokens/minute), **and** RPD (requests/day)
   limits at the same time. If you were tripping a per-day cap, no amount of
   sleeping between calls helps — that quota doesn't reset until it resets
   (usually ~24h), regardless of spacing.
3. **Retries can mask this and make it worse.** `ChatGoogleGenerativeAI` has
   its own internal retry/backoff on 429s. So a single `await` in your code
   can silently retry several times, burning through more of the same
   per-minute bucket you were trying to protect with the outer sleep — which
   is why it could look like "the delay isn't helping at all" rather than
   "the delay helped a little."

**The actual fix used below:** move the judge LLM to Groq, which has a much
larger free-tier requests-per-minute allowance than Gemini. That removes the
rate-limit pressure at the source instead of trying to schedule around it. A
tiny fixed pause is kept between calls purely as a courtesy buffer, not as
the fix.


In [11]:
async def run_evaluations(run, example):
    print("Evaluating Faithfulness...")
    faithfulness_score = await ragas_faithfulness(run, example)

    # Small courtesy buffer only - not the fix (Groq's free-tier RPM is the fix).
    await asyncio.sleep(2)

    print("Evaluating Response Relevancy...")
    relevancy_score = await ragas_response_relevancy(run, example)

    await asyncio.sleep(2)

    print("Evaluating ID-Based Precision & Recall...")
    # These two metrics are deterministic/ID-based and do not call any LLM,
    # so they don't touch the Groq or Gemini rate limits at all.
    precision_score = await ragas_context_precision_id_based(run, example)
    recall_score = await ragas_context_recall_id_based(run, example)

    return {
        "faithfulness": faithfulness_score,
        "response_relevancy": relevancy_score,
        "id_based_context_precision": precision_score,
        "id_based_context_recall": recall_score,
    }

# Execute evaluations
scores = await run_evaluations(result, reference_output)
print("\nFinal Evaluation Scores:")
print(scores)


NameError: name 'result' is not defined

In [15]:
print("Evaluating Faithfulness...")
faithfulness_score = await ragas_faithfulness(result, reference_output)
print(faithfulness_score)

Retrying langchain_google_genai.chat_models._achat_with_retry.<locals>._achat_with_retry in 2 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 29.344949386s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 20
}
, retry_d

Evaluating Faithfulness...


Retrying langchain_google_genai.chat_models._achat_with_retry.<locals>._achat_with_retry in 4 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 27.254277762s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 20
}
, retry_d

CancelledError: 

In [14]:
async def run_evaluations(run, example):
    print("Evaluating Faithfulness...")
    faithfulness_score = await ragas_faithfulness(run, example)

    # Small courtesy buffer only - not the fix (Groq's free-tier RPM is the fix).
    await asyncio.sleep(2)

    print("Evaluating Response Relevancy...")
    relevancy_score = await ragas_response_relevancy(run, example)

    await asyncio.sleep(2)

    print("Evaluating ID-Based Precision & Recall...")
    # These two metrics are deterministic/ID-based and do not call any LLM,
    # so they don't touch the Groq or Gemini rate limits at all.
    precision_score = await ragas_context_precision_id_based(run, example)
    recall_score = await ragas_context_recall_id_based(run, example)

    return {
        "faithfulness": faithfulness_score,
        "response_relevancy": relevancy_score,
        "id_based_context_precision": precision_score,
        "id_based_context_recall": recall_score,
    }

# Execute evaluations
scores = await run_evaluations(result, reference_output)
print("\nFinal Evaluation Scores:")
print(scores)


Evaluating Faithfulness...
Evaluating Response Relevancy...


BadRequestError: Error code: 400 - {'error': {'message': "'n' : number must be at most 1", 'type': 'invalid_request_error'}}

In [15]:
import asyncio
from ragas import RunConfig, SingleTurnSample
from ragas.metrics import (
    Faithfulness, 
    ResponseRelevancy, 
    IDBasedContextPrecision, 
    IDBasedContextRecall
)

# =========================================================
# 1. Define the RunConfig (The Fix for Rate Limiting)
# =========================================================
# This prevents Ragas from flooding your API with parallel requests
safe_config = RunConfig(max_workers=1, max_retries=5, max_wait=30)

# =========================================================
# 2. RAGAS Evaluation Functions
# =========================================================

async def ragas_faithfulness(run, example):
    sample = SingleTurnSample(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_context"],
    )
    scorer = Faithfulness(llm=ragas_llm)
    # Pass safe_config here to prevent parallel API flooding
    return await scorer.single_turn_ascore(sample, run_config=safe_config)


async def ragas_response_relevancy(run, example):
    sample = SingleTurnSample(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_context"]
    )
    # strictness=1 prevents Groq from throwing the "n must be at most 1" error
    scorer = ResponseRelevancy(
        llm=ragas_llm, 
        embeddings=ragas_embeddings,
        strictness=1 
    )
    # Pass safe_config here as well
    return await scorer.single_turn_ascore(sample, run_config=safe_config)


async def ragas_context_precision_id_based(run, example):
    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"],
    )
    scorer = IDBasedContextPrecision()
    # ID-based metrics do not use LLMs, so run_config is not strictly necessary here
    return await scorer.single_turn_ascore(sample)


async def ragas_context_recall_id_based(run, example):
    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"],
    )
    scorer = IDBasedContextRecall()
    return await scorer.single_turn_ascore(sample)

# =========================================================
# 3. Execution Pipeline (With safety delays)
# =========================================================

async def run_evaluations(run, example):
    print("Evaluating Faithfulness...")
    faithfulness_score = await ragas_faithfulness(run, example)
    
    # 2-second buffer between heavy metrics
    await asyncio.sleep(2) 

    print("Evaluating Response Relevancy...")
    relevancy_score = await ragas_response_relevancy(run, example)
    
    print("Evaluating ID-Based Precision & Recall...")
    precision_score = await ragas_context_precision_id_based(run, example)
    recall_score = await ragas_context_recall_id_based(run, example)

    return {
        "faithfulness": faithfulness_score,
        "response_relevancy": relevancy_score,
        "id_based_context_precision": precision_score,
        "id_based_context_recall": recall_score,
    }

# Assuming 'result' and 'reference_output' are already defined earlier in your notebook:
# scores = await run_evaluations(result, reference_output)
# print(scores)

In [16]:
async def run_evaluations(run, example):
    print("Evaluating Faithfulness...")
    faithfulness_score = await ragas_faithfulness(run, example)

    # Small courtesy buffer only - not the fix (Groq's free-tier RPM is the fix).
    await asyncio.sleep(2)

    print("Evaluating Response Relevancy...")
    relevancy_score = await ragas_response_relevancy(run, example)

    await asyncio.sleep(2)

    print("Evaluating ID-Based Precision & Recall...")
    # These two metrics are deterministic/ID-based and do not call any LLM,
    # so they don't touch the Groq or Gemini rate limits at all.
    precision_score = await ragas_context_precision_id_based(run, example)
    recall_score = await ragas_context_recall_id_based(run, example)

    return {
        "faithfulness": faithfulness_score,
        "response_relevancy": relevancy_score,
        "id_based_context_precision": precision_score,
        "id_based_context_recall": recall_score,
    }

# Execute evaluations
scores = await run_evaluations(result, reference_output)
print("\nFinal Evaluation Scores:")
print(scores)


Evaluating Faithfulness...


TypeError: SingleTurnMetric.single_turn_ascore() got an unexpected keyword argument 'run_config'

In [17]:
import asyncio
from ragas import RunConfig, SingleTurnSample
from ragas.metrics import (
    Faithfulness,
    ResponseRelevancy,
    IDBasedContextPrecision,
    IDBasedContextRecall
)

# Configuration for safe API usage (sequential, retries)
safe_config = RunConfig(max_workers=1, max_retries=5, max_wait=30)

async def ragas_faithfulness(run, example):
    sample = SingleTurnSample(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_context"],
    )
    scorer = Faithfulness(llm=ragas_llm, run_config=safe_config)
    return await scorer.single_turn_ascore(sample)

async def ragas_response_relevancy(run, example):
    sample = SingleTurnSample(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_context"]
    )
    scorer = ResponseRelevancy(
        llm=ragas_llm,
        embeddings=ragas_embeddings,
        strictness=1,
        run_config=safe_config
    )
    return await scorer.single_turn_ascore(sample)

async def ragas_context_precision_id_based(run, example):
    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"],
    )
    scorer = IDBasedContextPrecision()
    return await scorer.single_turn_ascore(sample)

async def ragas_context_recall_id_based(run, example):
    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"],
    )
    scorer = IDBasedContextRecall()
    return await scorer.single_turn_ascore(sample)

async def run_evaluations(run, example):
    print("Evaluating Faithfulness...")
    faithfulness_score = await ragas_faithfulness(run, example)
    await asyncio.sleep(2)   # buffer between heavy calls

    print("Evaluating Response Relevancy...")
    relevancy_score = await ragas_response_relevancy(run, example)
    await asyncio.sleep(2)

    print("Evaluating ID-Based Precision & Recall...")
    precision_score = await ragas_context_precision_id_based(run, example)
    recall_score = await ragas_context_recall_id_based(run, example)

    return {
        "faithfulness": faithfulness_score,
        "response_relevancy": relevancy_score,
        "id_based_context_precision": precision_score,
        "id_based_context_recall": recall_score,
    }

# Run the evaluations (assuming result and reference_output exist)
scores = await run_evaluations(result, reference_output)
print("\nFinal Evaluation Scores:")
print(scores)

Evaluating Faithfulness...


TypeError: Faithfulness.__init__() got an unexpected keyword argument 'run_config'